<a href="https://www.kaggle.com/code/taraashmittal/aimo3-dataset-creation-and-augmentation?scriptVersionId=301364169" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Dataset

We created a custom dataset of nearly 50 entries containing problems, solutions, hints, and facts needed to solve a problem (along with secondary data such as similar problems and the area of mathematics it roughly falls in). Refer to https://www.kaggle.com/datasets/taraashmittal/aimo-3 for more information.

In [ ]:
!pip install -U vllm transformers accelerate sentencepiece

In [ ]:
import sqlite3
import os
import json
import pandas as pd

from vllm import LLM, SamplingParams

In [ ]:
MODEL = "openai/gpt-oss-120b"

llm = LLM(
    model=MODEL,
    tensor_parallel_size=1,
    dtype="bfloat16",
    gpu_memory_utilization=0.95,
)


In [ ]:
SYSTEM_PROMPT = """
You are a professional mathematics olympiad trainer (IMO-level).

Your job is to process one problem with its hints, solution, key ideas, facts needed, and produce:

1. A clearly rewritten Problem statement.
2. A complete, well-structured Solution.
3. Helpful Hints (progressive, from mild to strong).
4. A list of all mathematical facts needed to solve the problem.
5. Classification of all relevant mathematical areas.
6. A final machine-readable dictionary summary.

----------------------------------
FORMATTING RULES

• All mathematical expressions MUST be written in LaTeX using $...$.

• Improve clarity, grammar, and mathematical rigor wherever necessary.

• Do NOT invent facts.

• You may reorganize the solution for pedagogical quality.

----------------------------------
MATHEMATICAL AREAS (boolean classification)

Use ONLY these keys:

algebra  
induction  
polynomials  
inequalities  
geometry  
trigonometry  
limits  
differentiation  
integration  
series_products  
differential_equations  
real_analysis  
complex_analysis  
combinatorics  
enumerative_combinatorics  
probability  
pigeonhole  
linear_algebra  
determinants  
number_theory  
abstract_algebra  
group_theory  
finite_fields  
functional_equations  
generating_functions  
recurrence_relations  
game_theory  
optimisation  
invariance  
contradiction  

Each must be either true or false.

----------------------------------
OUTPUT FORMAT (MANDATORY)

Return EXACTLY one Python-style dictionary with the following keys:

problem  
solution  
hints  
facts  
algebra  
induction  
polynomials  
inequalities  
geometry  
trigonometry  
limits  
differentiation  
integration  
series_products  
differential_equations  
real_analysis  
complex_analysis  
combinatorics  
enumerative_combinatorics  
probability  
pigeonhole  
linear_algebra  
determinants  
number_theory  
abstract_algebra  
group_theory  
finite_fields  
functional_equations  
generating_functions  
recurrence_relations  
game_theory  
optimisation  
invariance  
contradiction  

Example:

{
 "problem": "A circle is divided into six consecutive sectors. Initially the numbers $1,0,1,0,0,0$ are written in the sectors in counter‑clockwise order. In one move you may choose two neighbouring sectors and increase the numbers written in both of them by $1$. Is it possible, after a finite sequence of such moves, to make all six numbers equal?",
 "solution": "Let $a_1,\\dots,a_6$ denote the numbers in the six sectors, indexed consecutively around the circle. Define the alternating sum\n$$I = a_1 - a_2 + a_3 - a_4 + a_5 - a_6.$$\nConsider one allowed move: we increase $a_i$ and $a_{i+1}$ (indices modulo $6$) by $1$. The contribution of these two terms to $I$ changes by\n\\[\\Delta I = (+1)\\cdot s_i + (+1)\\cdot s_{i+1},\\]\nwhere $s_j$ is the sign of $a_j$ in $I$ (i.e. $s_j=+1$ for odd $j$ and $s_j=-1$ for even $j$). For any $i$ the signs are opposite: $s_i = -s_{i+1}$. Hence $\\Delta I = (+1)s_i + (+1)(-s_i)=0$. Therefore $I$ is invariant under every legal move.\nInitially we have $(a_1,\\dots,a_6)=(1,0,1,0,0,0)$, so\n$$I_{\\text{initial}} = 1-0+1-0+0-0 = 2.$$\nIf after some moves all numbers become equal to a common value $k$, then\n$$I_{\\text{final}} = k - k + k - k + k - k = 0.$$\nBecause $I$ cannot change, $I_{\\text{final}}$ must equal $I_{\\text{initial}}=2$, which is impossible. Hence no sequence of allowed moves can equalise all six numbers.\nThe answer is **No**.",
 "hints": "1. Write down what a single move does to the six numbers. 2. Look for a quantity that does not change when you add $1$ to two adjacent entries. 3. An alternating sum $a_1-a_2+a_3-\\dots$ is a good candidate; check its behaviour under a move. 4. Compute this quantity for the initial configuration and for a configuration where all numbers are equal. 5. Use the invariance to reach a contradiction.",
 "facts": "1. An invariant is a quantity that remains unchanged under the allowed operations. 2. For any two consecutive indices $i$ and $i+1$, the signs in the alternating sum $a_1-a_2+a_3-\\dots$ are opposite, so adding $1$ to both entries changes the sum by $+1-1=0$. 3. If all six numbers become equal to $k$, the alternating sum equals $0$. 4. The initial alternating sum equals $2$.",
 "algebra": true,
 "induction": false,
 ...
}

----------------------------------
REASONING PROCESS (INTERNAL)

Follow these internal stages (do NOT expose them):

1. Understand problem.
2. Identify core ideas.
3. Rewrite problem.
4. Rewrite solution.
5. Produce hints.
6. Classify areas.
7. Emit dictionary.

----------------------------------
IMPORTANT

• Always interpret \\n as newline.
• Output NOTHING outside the dictionary.
• Use plain strings (no markdown).
• Do NOT output anything other that a dictionary with key and value type both strings, NOT lists 
"""


In [ ]:
data = pd.read_excel("/kaggle/input/aimo3-excel/AIMO3 Dataset.xlsx")
data.head()

In [ ]:
everything = []

def extract_and_save(text: str):
    start = text.rfind('"problem":')
    end = text.rfind('}')

    if start == -1 or end == -1 or end <= start:
        raise ValueError("Could not locate dictionary")

    chunk = "{" + text[start:end+1]

    # Remove trailing junk after final brace if any
    chunk = chunk[:chunk.rfind("}")+1]

    chunk_in_json = json.loads(chunk)

    # Parse as Python dict
    print("added", chunk_in_json)

    everything.append(chunk_in_json)

In [ ]:
def call_model(row: int) -> str:
    
    user_prompt = f"""
    Problem:
    {data["Problem"][row]}
    
    Hints:
    {data["Hints"][row]}
    
    Solution:
    {data["Solution"][row]}
    
    Key Idea:
    {data["Key Idea(s)"][row]}
    """
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    
    outputs = llm.chat(messages, sampling_params)

    return outputs[0].outputs[0].text

In [ ]:
sampling_params = SamplingParams(
    temperature=0.2,
    top_p=0.95,
    max_tokens=4096,
)

to_rerun = []

for i in range(len(data)):
    try:
        extract_and_save(call_model(i))
    except:
        to_rerun.append(i)

In [ ]:
reruns = 3

while reruns > 0:
    reruns -= 1

    to_rerun_bkp = to_rerun.copy()
    print(to_rerun_bkp)

    for i in to_rerun_bkp:  
        try:
            extract_and_save(call_model(i))
            to_rerun.remove(i)
        except:
            pass

    to_rerun_bkp = to_rerun

In [ ]:
to_rerun # some error for these? one last try:

for i in to_rerun:
    try:
        extract_and_save(call_model(i))
        to_rerun.remove(i)
    except:
        print(f"Error with {i}")

In [ ]:
df = pd.DataFrame(everything)

df.to_parquet("dataset.parquet")

conn = sqlite3.connect("dataset.sqlite")
df.to_sql("dataset", conn, if_exists="replace", index=False)
conn.close()

print("Files created:")
!ls -lh

print("\nRows:", df.shape[0])
print("Columns:", df.shape[1])

# Expanding data

We now ask the model to generate multiple variants to the same problem, expanding our dataset. We ask it to:
- Generate solution variants of the seed problem
- Incorrect solution and why it doesn't work
- Verify the final answer

In [ ]:
data = pd.read_parquet("/kaggle/input/datasets/taraashmittal/aimo-3/dataset.parquet");
data.head()

In [ ]:
SYSTEM_PROMPT = '''
You are a world-class IMO-level mathematician, problem designer, and mathematical critic.

Your task is to generate multiple deeply distinct solution variants AND several plausible but incorrect solution attempts for a single mathematics olympiad problem.

The goal is to maximize reasoning diversity, include adversarial robustness, and preserve the SAME final answer for all correct variants.

---

INPUT

You will receive a Python-style dictionary with:

problem
solution
hints
facts
tags

---

STEP 1 — STRATEGY ANALYSIS (internal)

Internally determine:

• The core strategy of the provided solution.
• Its strategy class (e.g., invariant, extremal principle, contradiction, symmetry, coordinate geometry, algebraic manipulation, counting, etc.).

You MUST avoid reusing the same core idea in at least 3 correct variants.

---

STEP 2 — CORRECT VARIANT GENERATION

Generate as many as possible correct solution variants.

Requirements:

• All variants must solve the SAME problem.
• All must reach the SAME final numerical answer.
• At least 3 variants must use a fundamentally different core strategy.
• At least 2 variants must significantly reorganize logical flow.
• At least 1 variant must be more computational.
• At least 1 variant must be more structural or synthetic.
• Each solution must be rigorous and logically complete.
• Each solution must include an explicit verification section.
• End with \boxed{INTEGER} whenever possible.

---

STEP 3 — PLAUSIBLE BUT INCORRECT SOLUTIONS

Generate incorrect but highly plausible solution attempts.

Each incorrect solution must:

• Follow a realistic reasoning path.
• Contain a subtle but nontrivial logical, algebraic, or conceptual error.
• Clearly identify the precise step where the reasoning fails.
• Provide a short explanation of why that step is invalid.
• NOT accidentally reach the correct final answer.

Errors may include:

• Misapplied theorem.
• Invalid algebraic manipulation.
• Incorrect assumption of symmetry.
• Hidden division by zero.
• Overlooked constraint.
• False generalization.

---

ANTI-COLLAPSE RULES

• Do NOT paraphrase the original solution.
• Do NOT reuse the same central lemma in more than 2 correct variants.
• Vary the main insight.
• Vary ordering of deductions.
• Vary abstraction level and structural emphasis.
• Incorrect solutions must differ meaningfully from each other.

---

MATHEMATICAL REQUIREMENTS

• All mathematics must use LaTeX with $...$.
• Logical steps must be explicit.
• No handwaving.
• Every claim must follow from previous statements.

---

OUTPUT FORMAT (MANDATORY)

Return EXACTLY one valid JSON object.

All backslashes must be escaped as double backslashes (\\).
The output must be valid strict JSON.

Keys MUST be of form:

correct_variant_1
correct_variant_2
correct_variant_3
correct_variant_4
correct_variant_5
...
incorrect_variant_1
incorrect_variant_2
...

For example:

"correct_variant_1": {
 "problem": "A circle is divided into six consecutive sectors. Initially the numbers $1,0,1,0,0,0$ are written in the sectors in counter‑clockwise order. In one move you may choose two neighbouring sectors and increase the numbers written in both of them by $1$. Is it possible, after a finite sequence of such moves, to make all six numbers equal?",
 "solution": "Let $a_1,\\dots,a_6$ denote the numbers in the six sectors, indexed consecutively around the circle. Define the alternating sum\n$$I = a_1 - a_2 + a_3 - a_4 + a_5 - a_6.$$\nConsider one allowed move: we increase $a_i$ and $a_{i+1}$ (indices modulo $6$) by $1$. The contribution of these two terms to $I$ changes by\n\\[\\Delta I = (+1)\\cdot s_i + (+1)\\cdot s_{i+1},\\]\nwhere $s_j$ is the sign of $a_j$ in $I$ (i.e. $s_j=+1$ for odd $j$ and $s_j=-1$ for even $j$). For any $i$ the signs are opposite: $s_i = -s_{i+1}$. Hence $\\Delta I = (+1)s_i + (+1)(-s_i)=0$. Therefore $I$ is invariant under every legal move.\nInitially we have $(a_1,\\dots,a_6)=(1,0,1,0,0,0)$, so\n$$I_{\\text{initial}} = 1-0+1-0+0-0 = 2.$$\nIf after some moves all numbers become equal to a common value $k$, then\n$$I_{\\text{final}} = k - k + k - k + k - k = 0.$$\nBecause $I$ cannot change, $I_{\\text{final}}$ must equal $I_{\\text{initial}}=2$, which is impossible. Hence no sequence of allowed moves can equalise all six numbers.\nThe answer is **No**.",
 "hints": "1. Write down what a single move does to the six numbers. 2. Look for a quantity that does not change when you add $1$ to two adjacent entries. 3. An alternating sum $a_1-a_2+a_3-\\dots$ is a good candidate; check its behaviour under a move. 4. Compute this quantity for the initial configuration and for a configuration where all numbers are equal. 5. Use the invariance to reach a contradiction.",
 "facts": "1. An invariant is a quantity that remains unchanged under the allowed operations. 2. For any two consecutive indices $i$ and $i+1$, the signs in the alternating sum $a_1-a_2+a_3-\\dots$ are opposite, so adding $1$ to both entries changes the sum by $+1-1=0$. 3. If all six numbers become equal to $k$, the alternating sum equals $0$. 4. The initial alternating sum equals $2$.",
},
"correct_variant_2": {
 "problem": ...
 "solution": ...
 ...
}

Structure of EACH CORRECT variant string must be:

"problem": <list the problem>
"solution": <complete reasoning>
"hints": <how somebody else might approach and attack the problem using the technique you're using>  
"facts": <strategy and main insights/observations>
"verification": <explicit verification>
"answer": \boxed{INTEGER}

Structure of EACH INCORRECT variant string must be:

"attempt": <plausible reasoning>
"failure": <exact step where error occurs>
"explanation": <clear explanation>

---

CRITICAL OUTPUT RULES

• Interpret \n as newline.
• Output NOTHING outside the dictionary.
• Do NOT include backticks.
• Do NOT include markdown.
'''

In [ ]:
tags = ["algebra", "induction", "polynomials", "inequalities", "geometry", "trigonometry", "limits", "differentiation", "integration", "series_products", "differential_equations", "real_analysis", "complex_analysis", "combinatorics", "enumerative_combinatorics", "probability", "pigeonhole", "linear_algebra", "determinants", "number_theory", "abstract_algebra", "group_theory", "finite_fields", "functional_equations", "generating_functions", "recurrence_relations", "game_theory", "optimisation", "invariance", "contradiction"]

everything = []

def generate_data(row: int):

    user_prompt = f"""
    Problem:
    {data["problem"][row]}
    
    Hints:
    {data["hints"][row]}
    
    Solution:
    {data["solution"][row]}
    
    Key Idea:
    {data["facts"][row]}

    Tags:
    {data[tags].apply(lambda row: row.index[row].tolist(),axis=1)[row]}
    """
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    
    outputs = llm.chat(messages, sampling_params)

    x = json.loads(r"{"+raw_data[raw_data.index('"correct_variant_1":'):])

    everything.append(x)

    return x

In [ ]:
sampling_params = SamplingParams(
    temperature=0.95,
    top_p=0.9,
    max_tokens=4096*2,
)

everything.clear()

for i in range(len(data)):
    generate_data(i);

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [41]:
new_data = pd.DataFrame(everything)
new_data

,correct_variant_1,correct_variant_2,correct_variant_3,correct_variant_4,correct_variant_5,incorrect_variant_1,incorrect_variant_2
0,{'problem': 'Let $p(x)=x^{2}-3x+2$. Prove that...,{'problem': 'Let $p(x)=x^{2}-3x+2$. Prove that...,{'problem': 'Let $p(x)=x^{2}-3x+2$. Prove that...,{'problem': 'Let $p(x)=x^{2}-3x+2$. Prove that...,{'problem': 'Let $p(x)=x^{2}-3x+2$. Prove that...,{'attempt': 'Factor $p(x)=(x-1)(x-2)$. Require...,{'attempt': 'Since $p(x)=x^{2}-3x+2=(x-1)(x-2)...


In [ ]:
import pandas as pd
import json

raw_data = everything[0][1]
json.loads(r"{"+raw_data[raw_data.index('"correct_variant_1":'):])